# 01 — Exploratory Data Analysis

Pneumonia Detection from Chest X-Rays.

Covers: dataset statistics, class imbalance, image dimensions, pixel intensity distribution, sample visualization, mean/std, and corrupted-image detection (Phase 1 of the spec).

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

from config import CONFIG
from src.utils import count_images
from src.preprocessing import validate_image, load_image

%matplotlib inline

## Dataset statistics & class imbalance

In [ ]:
splits = {'train': CONFIG.paths.train_dir, 'val': CONFIG.paths.val_dir, 'test': CONFIG.paths.test_dir}
rows = []
for split, path in splits.items():
    counts = count_images(path)
    for cls, n in counts.items():
        rows.append({'split': split, 'class': cls, 'count': n})

df = pd.DataFrame(rows)
print(df)
if not df.empty:
    df.pivot(index='split', columns='class', values='count').plot(kind='bar', figsize=(7,4))
    plt.title('Class distribution per split')
    plt.ylabel('Number of images')
    plt.show()
else:
    print('No images found yet — download the dataset into dataset/train|val|test/NORMAL|PNEUMONIA first.')

## Corrupted / unreadable image detection

In [ ]:
def scan_for_corrupted(directory):
    directory = Path(directory)
    bad = []
    if not directory.exists():
        return bad
    for f in directory.rglob('*'):
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
            if not validate_image(f):
                bad.append(str(f))
    return bad

bad_files = scan_for_corrupted(CONFIG.paths.dataset_dir)
print(f'Corrupted/unreadable images found: {len(bad_files)}')
bad_files[:10]

## Image dimensions

In [ ]:
dims = []
train_normal = CONFIG.paths.train_dir / 'NORMAL'
if train_normal.exists():
    for f in list(train_normal.iterdir())[:200]:
        img = cv2.imread(str(f))
        if img is not None:
            dims.append(img.shape[:2])

if dims:
    dims_arr = np.array(dims)
    print('Height: mean', dims_arr[:,0].mean(), 'std', dims_arr[:,0].std())
    print('Width:  mean', dims_arr[:,1].mean(), 'std', dims_arr[:,1].std())
    plt.figure(figsize=(6,4))
    plt.scatter(dims_arr[:,1], dims_arr[:,0], alpha=0.4)
    plt.xlabel('Width'); plt.ylabel('Height'); plt.title('Image dimension scatter (sample)')
    plt.show()

## Pixel intensity histogram

In [ ]:
intensities = []
if train_normal.exists():
    for f in list(train_normal.iterdir())[:100]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            intensities.append(img.ravel())

if intensities:
    all_pixels = np.concatenate(intensities)
    plt.figure(figsize=(7,4))
    plt.hist(all_pixels, bins=50, color='steelblue')
    plt.title('Pixel intensity distribution (sample, NORMAL class)')
    plt.xlabel('Pixel value'); plt.ylabel('Frequency')
    plt.show()
    print('Mean pixel value:', all_pixels.mean(), '| Std:', all_pixels.std())

## Sample visualization

In [ ]:
def show_samples(class_dir, n=6, title=''):
    class_dir = Path(class_dir)
    if not class_dir.exists():
        print('Missing:', class_dir)
        return
    files = list(class_dir.iterdir())[:n]
    fig, axes = plt.subplots(1, len(files), figsize=(3*len(files), 3))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, files):
        img = load_image(f)
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

show_samples(CONFIG.paths.train_dir / 'NORMAL', title='NORMAL samples')
show_samples(CONFIG.paths.train_dir / 'PNEUMONIA', title='PNEUMONIA samples')

## Summary

Record observed class imbalance, resolution range, and any corrupted files here before moving to preprocessing.